# LLM Intro — Plain Text Completion & Streaming

Compare how **OpenAI**, **Anthropic**, **Google Gemini**, and **Ollama** (local) handle a simple text prompt and streaming output using each provider's native Python SDK.

In [11]:
import os

PROMPT = 'Create a random number.'

---
## OpenAI

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
openai_resp = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)
print(openai_resp.choices[0].message.content)

684237519


---
## Anthropic

In [3]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-6'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
anthropic_resp = anthropic_client.messages.create(
    model=ANTHROPIC_MODEL,
    max_tokens=300,
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)
print(anthropic_resp.content[0].text)

Here's a random number:

**47**

(Generated arbitrarily — let me know if you need a number within a specific range or format!)


---
## Google Gemini

In [ ]:
from google import genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

client = genai.Client(api_key=GEMINI_API_KEY)
gemini_resp = client.models.generate_content(
    model=GOOGLE_MODEL,
    contents=PROMPT
)
print(gemini_resp.text)

Here's a random number:

**42**

I generated an integer between 1 and 100. Let me know if you'd like another one with different parameters (e.g., a decimal, a specific range)!


---
## Ollama (local)

In [5]:
import ollama

OLLAMA_MODEL = 'llama3.2:3b-instruct-q5_K_M'

# Discover locally installed Ollama models:
# for m in ollama.list().models:
#     print(m.model)

ollama_resp = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[
        {
            'role': 'user',
            'content': PROMPT
        }
    ]
)
print(ollama_resp['message']['content'])

The random number I generated is: **854**


---

# Streaming — Token-by-Token Output

Instead of waiting for the full response, **streaming** lets you display tokens as they are generated — critical for responsive UIs and long outputs.

Each provider uses a slightly different streaming API, but the pattern is the same: open a stream, iterate chunks, print incrementally.

In [13]:
PROMPT = 'Write a short poem about machine learning in exactly 12 lines.'

---
## OpenAI

Set `stream=True`; iterate over chunks and read `.choices[0].delta.content`.

In [14]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
stream = openai_client.chat.completions.create(
    model=OPENAI_MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True
)
for chunk in stream:
    token = chunk.choices[0].delta.content
    if token:
        print(token, end='', flush=True)
print()

Data hums in grids of dawn and dusk,
Labels whisper, partial, sometimes wrong;
Gradients flow like rivers, sure and strong;
Loss is a bruise the model learns to soothe;
Weights shift, a murmuration finding truth.
Features bloom from noise, a hidden choir;
Bias lurks—a shadow stitched to wire;
We prune, apply dropout, cross-validate;
Patience backpropagates through every state;
On test-time hills, the oracle is shy,
Yet forecasts kindle sparks behind the eye;
Machine and mind co-train toward the sky.


---
## Anthropic

Use `client.messages.stream()` as a context manager; iterate `.text_stream` for token strings.

In [8]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-6'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
with anthropic_client.messages.stream(
    model=ANTHROPIC_MODEL,
    max_tokens=300,
    messages=[{'role': 'user', 'content': PROMPT}]
) as stream:
    for token in stream.text_stream:
        print(token, end='', flush=True)
print()

Here is a four-line poem about machine learning:

Data flows through layers deep,
Patterns learned while humans sleep,
Weights adjust with every turn,
Teaching machines the way to learn.


---
## Google Gemini

Call `client.models.generate_content_stream(...)`; each chunk has a `.text` attribute.

In [9]:
from google import genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

client = genai.Client(api_key=GEMINI_API_KEY)
for chunk in client.models.generate_content_stream(model=GOOGLE_MODEL, contents=PROMPT):
    print(chunk.text, end='', flush=True)
print()

From vast data, patterns it discerns,
A web of logic, new knowledge earns.
Predicting futures, with careful grace,
Improving always, its silicon trace.


---
## Ollama (local)

Set `stream=True`; each chunk is a dict; read `chunk['message']['content']`.

In [10]:
import ollama

OLLAMA_MODEL = 'llama3.2:3b-instruct-q5_K_M'

for chunk in ollama.chat(
    model=OLLAMA_MODEL,
    messages=[{'role': 'user', 'content': PROMPT}],
    stream=True
):
    print(chunk['message']['content'], end='', flush=True)
print()

In silicon halls, data reigns
Algorithms dance, and insights gain
Machine learning's subtle art
Weaves patterns, a digital heart
